# A-B05: 뇌동맥류 이진 nnU-Net Focal Tversky + Component-centered Sampling

기존 팀원용 동맥류 폴더와 분리된 Dataset503 이진 분할 baseline입니다.

- 데이터: `Dataset503_TopAneuMRABinary` (background=0, aneurysm=1)
- 모델: standard nnU-Net `3d_fullres`
- loss: Focal Tversky (Tversky α=0.3, β=0.7; Focal γ=2.0)
- optimizer: SGD
- sampling: foreground patch에서 connected component 균등 선택
- 학습: 300 epochs, fold 0
- epoch 195부터 매 epoch checkpoint 보관, 200 epoch부터 10 epoch마다 full-volume validation
- macro Dice Top-3와 각 ±5 epoch checkpoint 보존 및 종료 후 재검증
- 결과: `runs/aneurysm_binary/nnunet`
- 다시 실행하면 `checkpoint_latest.pth`에서 자동 재개합니다.

## 1. 독립 경로 설정

In [1]:
from pathlib import Path
import json
import subprocess
import time

ROOT = Path("/home/stu03/workspace/model")
CONFIG = ROOT / "configs/experiments/aneurysm_binary/A-B05.yaml"
DATA_ROOT = Path("/home/stu03/workspace/datasets/aneurysm/v2/model_ready/nnunet")
RESULTS_ROOT = ROOT / "runs/aneurysm_binary/nnunet"
PYTHON_BIN = ROOT / ".venv/bin/python"
LAUNCHER = ROOT / "scripts/aneurysm_binary/training/run_a_b05.py"

DATASET_NAME = "Dataset503_TopAneuMRABinary"
TRAINER = "nnUNetTrainerAneurysmBinaryBaseline"
FOLD_DIR = RESULTS_ROOT / DATASET_NAME / f"{TRAINER}__AneurysmBinaryPlansTverskyFocalComponentCenteredV1__3d_fullres/fold_0"
PROGRESS_FILE = FOLD_DIR / "progress.json"
LOG_FILE = ROOT / "runs/aneurysm_binary/A-B05/train.log"
PID_FILE = ROOT / "runs/aneurysm_binary/A-B05/train.pid"

for name, path in {
    "config": CONFIG,
    "python": PYTHON_BIN,
    "launcher": LAUNCHER,
    "preprocessed": DATA_ROOT / "nnUNet_preprocessed" / DATASET_NAME,
}.items():
    print(f"{name:12s}: {path.exists()}  {path}")

config      : True  /home/stu03/workspace/model/configs/experiments/aneurysm_binary/A-B05.yaml
python      : True  /home/stu03/workspace/model/.venv/bin/python
launcher    : True  /home/stu03/workspace/model/scripts/aneurysm_binary/training/run_a_b05.py
preprocessed: True  /home/stu03/workspace/datasets/aneurysm/v2/model_ready/nnunet/nnUNet_preprocessed/Dataset503_TopAneuMRABinary


## 2. 실행 전 dry-run

파일을 만들거나 GPU를 사용하지 않고, 실행 명령과 출력 폴더만 확인합니다.

In [ ]:
base_command = [
    str(PYTHON_BIN), str(LAUNCHER),
    "--config", str(CONFIG),
    "--data-root", str(DATA_ROOT),
    "--results-root", str(RESULTS_ROOT),
    "--python-bin", str(PYTHON_BIN),
]
subprocess.run(base_command + ["--dry-run"], cwd=ROOT, check=True)

## 3. GPU 확인

In [ ]:
subprocess.run(["nvidia-smi"], check=False)

## 4. 학습 시작 또는 재개

이 셀은 학습을 백그라운드로 시작하므로 다음 진행률 셀을 바로 실행할 수 있습니다. 기존 `checkpoint_latest.pth`가 있으면 launcher가 자동으로 `--c`를 붙여 재개합니다. 이미 실행 중인 PID가 있으면 중복 실행하지 않습니다.

In [ ]:
LOG_FILE.parent.mkdir(parents=True, exist_ok=True)

def pid_is_running(pid_file):
    if not pid_file.is_file():
        return False
    try:
        pid = int(pid_file.read_text().strip())
        return subprocess.run(["kill", "-0", str(pid)], capture_output=True).returncode == 0
    except (ValueError, OSError):
        return False

if pid_is_running(PID_FILE):
    print("이미 실행 중입니다. PID:", PID_FILE.read_text().strip())
else:
    log_handle = LOG_FILE.open("a", encoding="utf-8")
    process = subprocess.Popen(
        base_command,
        cwd=ROOT,
        stdout=log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    PID_FILE.write_text(str(process.pid), encoding="utf-8")
    print("A-B05 시작 PID:", process.pid)
    print("로그:", LOG_FILE)

## 5. 실시간 진행률

중단하려면 이 셀의 정지 버튼만 누르세요. 학습 프로세스는 계속 실행됩니다.

In [3]:
from IPython.display import clear_output, display
import pandas as pd

def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def format_time(seconds):
    if seconds is None:
        return "-"
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def pid_is_running(pid_file):
    if not pid_file.is_file():
        return False
    try:
        pid = int(pid_file.read_text(encoding="utf-8").strip())
        return subprocess.run(
            ["kill", "-0", str(pid)], capture_output=True
        ).returncode == 0
    except (ValueError, OSError):
        return False

while True:
    clear_output(wait=True)
    p = read_json(PROGRESS_FILE)
    running = pid_is_running(PID_FILE)
    status = p.get("status", "starting" if running else "not_started")
    row = {
        "실험": "A-B05",
        "상태": status,
        "Epoch": f"{p.get('current_epoch', 0)}/{p.get('total_epochs', 300)}",
        "진행률(%)": p.get("progress_percent", 0),
        "ETA": format_time(p.get("eta_seconds")),
        "Train Loss": p.get("train_loss"),
        "Val Loss": p.get("val_loss"),
        "Dice": p.get("pseudo_dice"),
        "Recall": p.get("pseudo_recall"),
        "Composite": p.get("composite"),
        "Best Dice": p.get("best_dice"),
        "Best Recall": p.get("best_recall"),
        "Best Composite": p.get("best_composite"),
        "Full Macro Dice": p.get("full_macro_dice"),
        "Best Full Macro Dice": p.get("best_full_macro_dice"),
        "Best Epoch": p.get("best_full_macro_epoch"),
        "Top-3 Epoch": p.get("top_full_macro_epochs"),
        "보존 checkpoint": p.get("retained_epoch_checkpoints"),
        "PID 실행 중": running,
    }
    display(pd.DataFrame([row]))
    print("로그 마지막 15줄:")
    if LOG_FILE.is_file():
        print("".join(LOG_FILE.read_text(encoding="utf-8", errors="replace").splitlines(True)[-15:]))
    if status == "completed" or (not running and p):
        print("학습 프로세스가 종료되었습니다. 아래 결과 확인 셀을 실행하세요.")
        break
    time.sleep(10)

,실험,상태,Epoch,진행률(%),ETA,Train Loss,Val Loss,Dice,Recall,Composite,Best Dice,Best Recall,Best Composite,Full Macro Dice,Best Full Macro Dice,Best Epoch,Top-3 Epoch,보존 checkpoint,PID 실행 중
0,A-B05,completed,300/300,100.0,00:00:00,-0.502035,None,0.77519,0.728,0.751595,0.848948,0.933184,0.864411,0.462656,0.504252,280,"[280, 270, 230]",37,True


로그 마지막 15줄:
학습 프로세스가 종료되었습니다. 아래 결과 확인 셀을 실행하세요.


## 6. 결과와 checkpoint 확인

In [4]:
print("결과 폴더:", FOLD_DIR)
if not FOLD_DIR.is_dir():
    print("아직 결과 폴더가 없습니다.")
else:
    for path in sorted(FOLD_DIR.iterdir()):
        if path.is_file():
            print(f"{path.name:35s} {path.stat().st_size / 1024**2:9.2f} MB")

summary = FOLD_DIR / "validation" / "summary.json"
print("Full-volume validation summary:", summary.exists(), summary)

결과 폴더: /home/stu03/workspace/model/runs/aneurysm_binary/nnunet/Dataset503_TopAneuMRABinary/nnUNetTrainerAneurysmBinaryBaseline__AneurysmBinaryPlansTverskyFocalComponentCenteredV1__3d_fullres/fold_0
best_full_macro.json                     0.00 MB
checkpoint_best.pth                    235.09 MB
checkpoint_best_composite.pth          235.09 MB
checkpoint_best_dice.pth               235.09 MB
checkpoint_best_dual_criteria.pth      235.09 MB
checkpoint_best_full_macro.pth         235.09 MB
checkpoint_best_recall.pth             235.07 MB
checkpoint_final.pth                   235.09 MB
component_sampling.json                  0.04 MB
custom_metrics.json                      0.00 MB
debug.json                               0.02 MB
final_selection.json                     0.01 MB
full_validation_history.json             0.02 MB
ich_config.json                          0.00 MB
progress.json                            0.00 MB
progress.png                             1.23 MB
training_log_2026_

## 7. 재개 방법

학습이 서버 종료나 OOM으로 중단되면 **4번 셀을 다시 실행**하면 됩니다. `checkpoint_latest.pth`가 존재할 때 자동 재개합니다. 새로 처음부터 돌리고 싶을 때만 기존 결과 폴더를 별도로 백업한 뒤 `--no-resume`을 사용하세요.

## 8. Full-volume 상위 후보와 최종 모델 선택

학습 종료 후 launcher가 기존 방식으로 full-volume macro Dice 상위 3개와 각 주변 ±5 epoch를 평가합니다. 최종 macro Dice 상위 3개에는 저장된 예측을 이용한 병변 분석을 추가합니다.

- 최고 macro Dice에서 `0.01` 이내인 후보만 유지
- 그중 병변 탐지율이 가장 높은 checkpoint 선택
- 탐지율이 같으면 macro Dice가 높은 checkpoint 선택

최종 모델은 `checkpoint_best_dual_criteria.pth`로 저장됩니다.

In [5]:
# 후보별 macro Dice와 병변 탐지 성능
import pandas as pd

SELECTION_FILE = FOLD_DIR / "final_selection.json"
if not SELECTION_FILE.is_file():
    print("아직 최종 선택 결과가 없습니다. 학습 및 후처리 완료 후 다시 실행하세요.")
else:
    selection = read_json(SELECTION_FILE)
    selected_epoch = int(selection["selected"]["epoch"])
    best_macro = float(selection["best_macro_dice"])
    tolerance = 0.01
    rows = []
    for item in selection["candidates"]:
        rows.append({
            "epoch": int(item["epoch"]),
            "positive_macro_dice": float(item["macro_dice_positive"]),
            "lesion_detection_rate": float(item["lesion_detection_rate"]),
            "detected/total": f'{int(item["detected_lesions"])}/{int(item["total_lesions"])}',
            "mean_lesion_dice": float(item["mean_lesion_dice"]),
            "mean_dice_detected_only": float(item["mean_dice_detected_only"]),
            "macro_0.01_이내": float(item["macro_dice_positive"]) >= best_macro - tolerance,
            "최종선택": int(item["epoch"]) == selected_epoch,
        })
    candidate_df = pd.DataFrame(rows).sort_values("positive_macro_dice", ascending=False)
    display(candidate_df.style.format({
        "positive_macro_dice": "{:.4f}",
        "lesion_detection_rate": "{:.2%}",
        "mean_lesion_dice": "{:.4f}",
        "mean_dice_detected_only": "{:.4f}",
    }))
    print("선택 규칙:", selection["rule"])
    print("최종 epoch:", selected_epoch)
    print("최종 checkpoint:", FOLD_DIR / "checkpoint_best_dual_criteria.pth")

,epoch,positive_macro_dice,lesion_detection_rate,detected/total,mean_lesion_dice,mean_dice_detected_only,macro_0.01_이내,최종선택
0,285,0.5088,75.00%,30/40,0.4977,0.6637,True,False
1,279,0.5052,77.50%,31/40,0.5687,0.7338,True,False
2,280,0.5043,80.00%,32/40,0.5307,0.6634,True,True


선택 규칙: macro_dice_positive within 0.010000 of best; maximize lesion_detection_rate; tie-break by macro_dice_positive
최종 epoch: 280
최종 checkpoint: /home/stu03/workspace/model/runs/aneurysm_binary/nnunet/Dataset503_TopAneuMRABinary/nnUNetTrainerAneurysmBinaryBaseline__AneurysmBinaryPlansTverskyFocalComponentCenteredV1__3d_fullres/fold_0/checkpoint_best_dual_criteria.pth


## 9. 최종 선택 모델의 병변 크기별 결과

In [6]:
if not SELECTION_FILE.is_file():
    print("아직 최종 선택 결과가 없습니다.")
else:
    lesion_dir = FOLD_DIR / "lesion_analysis" / f"epoch_{selected_epoch:04d}"
    size_summary_file = lesion_dir / "lesion_size_summary.csv"
    lesion_summary_file = lesion_dir / "lesion_summary.json"
    lesion_summary = read_json(lesion_summary_file)
    print("전체 병변:", lesion_summary.get("total_lesions"))
    print("탐지 병변:", lesion_summary.get("detected_lesions"))
    print("병변 탐지율:", f'{lesion_summary.get("lesion_detection_rate", float("nan")):.4f}')
    print("전체 병변 평균 Dice:", f'{lesion_summary.get("mean_lesion_dice", float("nan")):.4f}')
    print("탐지 병변 평균 Dice:", f'{lesion_summary.get("mean_dice_detected_only", float("nan")):.4f}')
    display(pd.read_csv(size_summary_file))

전체 병변: 40
탐지 병변: 32
병변 탐지율: 0.8000
전체 병변 평균 Dice: 0.5307
탐지 병변 평균 Dice: 0.6634


,size_group,gt_lesions,detected_lesions,detection_rate,mean_lesion_dice,median_lesion_dice,mean_dice_detected_only,median_volume_ml,detected/total
0,<0.01 mL,7,1,0.142857,0.046163,0.000000,0.323144,0.005115,1/7
1,0.01–0.05 mL,21,20,0.952381,0.621845,0.720000,0.652938,0.022060,20/21
2,0.05–0.1 mL,4,3,0.750000,0.556981,0.681178,0.742641,0.069179,3/4
3,0.1–0.5 mL,3,3,1.000000,0.698542,0.827146,0.698542,0.233645,3/3
4,≥0.5 mL,5,5,1.000000,0.704400,0.828288,0.704400,1.498535,5/5
